# Fine-tuning Embeddings and Advanced Retrieval

## Overview

- Select an embedding model for fine-tuning

- Adjust the input dataset to the format expected by the S-BERT/Hugging Face: `triplets = {anchor, positives, negatives}`

- Choose a training loss and trains the model with the new datasets

- Use MLFlow to log and register the trained model

- Create an endpoint for the fine-tuned model and deploy it

- Test the deployed endpoint

## Setups

In [0]:
%sql
-- setup to catalog and schema
use catalog `studies`;
use schema `databricks_finetuning`;

In [0]:
import warnings
warnings.filterwarnings('ignore')

from typing import List, Callable
from rich import print
import os
import tempfile

import pandas as pd
import numpy as np

import mlflow
from mlflow.tracking.client import MlflowClient

from datasets import Dataset
import sentence_transformers

import torch

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

In [0]:
EXPORT_EVAL_SET_PREDS_FT = True
EVAL_SET_PREDS_FT = 'eval_set_preds_ft'

VS_FT_ENDPOINT_NAME = 'ft_article_vs_endpoint'
VS_FT_INDEX_NAME = 'article_vs_index_ft'
CHUNKS_TABLE_NAME = 'article_chunks'
OUTPUT_TABLE_NAME = 'generated_questions_llm'

REGISTERED_FT_MODEL_NAME = 'ft_embedding_model_endpoint'  # 'ft_embedding_model'
EVAL_SET_PREDS = 'eval_set_preds'

HF_EMBEDDING_MODELS = {
    'gte-small': 'thenlper/gte-small',  # cpu
    'all-MiniLM-L6-v1': 'sentence-transformers/all-MiniLM-L6-v1',  # cpu
    'all-MiniLM-L6-v2': 'sentence-transformers/all-MiniLM-L6-v2',  # cpu
    'bge-small-en-v1.5': 'BAAI/bge-small-en-v1.5',  # cpu
    'bge-large-en': 'BAAI/bge-large-en',  # gpu
}

TRAINING_LOSS = 'MNRL'  
# MNRL (MultipleNegativesRankingLoss), CSL (CosineSimilarityLoss), CMNRL (CachedMultipleNegativesRankingLoss)

infos_env = spark.sql('SELECT current_catalog(), current_schema()').collect()[0]
CATALOG = infos_env[0]
SCHEMA = infos_env[1]
print(f'Catalog: {CATALOG}')
print(f'Schema: {SCHEMA}')


RUN_TRAIN_MODEL = False
RUN_DEPLOY_MODEL = False
CREATE_FT_VS_INDEX = False
RUN_CREATE_EMBED_CHUNKS = True
TEST_FT_MODEL_LATEST_VERSION = False


REDUCE_DATASETS = False
N_MAX = 1000
NUM_TRAIN_EPOCHS = 3
EVAL_STEPS_TRAINING = 1

Catalog: workspace
Schema: default


In [0]:
cuda_is_available = torch.cuda.is_available()
print(cuda_is_available)

False

## Base model for embedding

In [0]:
# https://huggingface.co/models?other=base_model:quantized:thenlper/gte-small
embed_model_name = HF_EMBEDDING_MODELS['gte-small']

embed_model = sentence_transformers.SentenceTransformer(embed_model_name, device='cpu')
# embed_model.to('cpu')

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
print('=== Checking the base model ===')
data = ['Helo world!']
embeddings = embed_model.encode(data)
print(f'Input data: {data}')
print(f'Embeddings shape: {embeddings.shape}')
print(embeddings[0][:10])

=== Checking the base model ===

Input data: ['Helo world!']

Embeddings shape: (1, 384)

[-0.01913   0.002438  0.0846   -0.0273    0.007397  0.0232    0.08417
  0.0341   -0.02458  -0.02814 ]

## Prepare the datasets for fine-tuning

In [0]:
train_set = spark.table(f'{CATALOG}.{SCHEMA}.{OUTPUT_TABLE_NAME}_train').toPandas()
eval_set = spark.table(f'{CATALOG}.{SCHEMA}.{OUTPUT_TABLE_NAME}_eval').toPandas()

print(f'train_set.shape: {train_set.shape}')
display(train_set.head())
print(f'\neval_set.shape: {eval_set.shape}')
eval_set.head()

train_set.shape: (58, 4)

path,inputs,id,preds
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,We present a model of online content sharing where agents sequentially observe an article and must decide whether to share it with others. This content may or may not contain misinformation. Agents gain utility from positive social media interactions but do not want to be called out for propagating,1,'How do agents' motivations for positive social media interactions influence the spread of misinformation in the proposed model?'
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,be called out for propagating misinformation. We characterize the (Bayesian-Nash) equilibria of this social media game and show sharing exhibits strategic complementarity. Our first main result establishes that the impact of homophily on content virality is non-monotone: homophily reduces the broad,2,'How does homophily non-monotonically affect content virality in the Bayesian-Nash equilibrium of the social media sharing game?'
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"nd when there is greater polarization and more divisive content. Finally, we discuss various regulatory solutions to such platform-manufactured misinformation.",5,'What regulatory solutions are proposed to address platform-manufactured misinformation in the context of increased polarization and divisive content?'
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pg2_abstract.pdf,"ximizing engagement tend to design their algorithms to create more homophilic communication patterns (""filter bubbles""). We show that platform incentives to amplify misinformation are particularly pronounced for low-reliability content likely to contain misinformation and when there is greater polar",4,"'Why do platforms incentivize the amplification of low-reliability content, and how does polarization influence this dynamic?'"
dbfs:/Volumes/studies/databricks_finetuning/article_infos/pgs19_20_regulation.pdf,"(c) If $\rho > p^*$ , the provenance policy reduces the virality of misinformation even when the platform optimally chooses the sharing network conditional on the policy.",34,"'What condition ensures that the provenance policy reduces the virality of misinformation, even under optimal platform-driven network selection?'"


eval_set.shape: (25, 4)

,path,inputs,id,preds
0,dbfs:/Volumes/studies/databricks_finetuning/ar...,6 Regulation\nOur analysis so far raises the n...,21,'What types of regulations does the paper sugg...
1,dbfs:/Volumes/studies/databricks_finetuning/ar...,We consider the effects of these policies both...,25,'How does the paper model the regulator’s stra...
2,dbfs:/Volumes/studies/databricks_finetuning/ar...,misinformation below a given threshold; (4) ne...,24,'How do network regulations aim to mitigate th...
3,dbfs:/Volumes/studies/databricks_finetuning/ar...,and recommends articles to users with aligned ...,58,'How do algorithmic recommendations contribute...
4,dbfs:/Volumes/studies/databricks_finetuning/ar...,"f misinformation, agents will be uncertain abo...",69,'How does misinformation affect agents' abilit...


**Note:** Training dataset S-BERT = {triplets: triplets = (anchor, positive example, negative example)}

In [0]:
cols_name = {'inputs': 'anchor', 'preds': 'positive'}
# the negative example will be automatic generated by the training loss

if REDUCE_DATASETS:
    train_set_reduced =pd.DataFrame({})
    eval_set_reduced =pd.DataFrame({})

    for path_i in train_set.path.unique():
        row_i = train_set[train_set.path == path_i].sample(1)
        train_set_reduced = pd.concat([train_set_reduced, row_i])
    
    for path_i in eval_set.path.unique():
        row_i = eval_set[eval_set.path == path_i].sample(1)
        eval_set_reduced = pd.concat([eval_set_reduced, row_i])
    
    ft_train_dataset = Dataset.from_pandas(train_set_reduced.rename(columns=cols_name), preserve_index=False)
    ft_eval_dataset = Dataset.from_pandas(eval_set_reduced.rename(columns=cols_name), preserve_index=False)
    
else:
    ft_train_dataset = Dataset.from_pandas(train_set.rename(columns=cols_name), preserve_index=False)
    ft_eval_dataset = Dataset.from_pandas(eval_set.rename(columns=cols_name), preserve_index=False)


ft_train_dataset = ft_train_dataset.select_columns(['anchor', 'positive'])
ft_eval_dataset = ft_eval_dataset.select_columns(['anchor', 'positive'])

print(f'ft_train_dataset.num_rows: {ft_train_dataset.num_rows}')
display(ft_train_dataset.to_pandas())

print(f'ft_eval_dataset.num_rows: {ft_eval_dataset.num_rows}')
ft_eval_dataset.to_pandas()

ft_train_dataset.num_rows: 58

anchor,positive
We present a model of online content sharing where agents sequentially observe an article and must decide whether to share it with others. This content may or may not contain misinformation. Agents gain utility from positive social media interactions but do not want to be called out for propagating,'How do agents' motivations for positive social media interactions influence the spread of misinformation in the proposed model?'
be called out for propagating misinformation. We characterize the (Bayesian-Nash) equilibria of this social media game and show sharing exhibits strategic complementarity. Our first main result establishes that the impact of homophily on content virality is non-monotone: homophily reduces the broad,'How does homophily non-monotonically affect content virality in the Bayesian-Nash equilibrium of the social media sharing game?'
"nd when there is greater polarization and more divisive content. Finally, we discuss various regulatory solutions to such platform-manufactured misinformation.",'What regulatory solutions are proposed to address platform-manufactured misinformation in the context of increased polarization and divisive content?'
"ximizing engagement tend to design their algorithms to create more homophilic communication patterns (""filter bubbles""). We show that platform incentives to amplify misinformation are particularly pronounced for low-reliability content likely to contain misinformation and when there is greater polar","'Why do platforms incentivize the amplification of low-reliability content, and how does polarization influence this dynamic?'"
"(c) If $\rho > p^*$ , the provenance policy reduces the virality of misinformation even when the platform optimally chooses the sharing network conditional on the policy.","'What condition ensures that the provenance policy reduces the virality of misinformation, even under optimal platform-driven network selection?'"
19 See Vanity Fair: https://www.vanityfair.com/news/2020/12/with-the-election-over-facebook-gets-back-to-spreading-misinformation and also https://www.technologyreview.com/2021/03/11/1020600/facebook-responsible-ai-misinformation/. 17,"'How did Facebook resume spreading misinformation after the 2020 election, according to the Vanity Fair and MIT Technology Review articles?'"
"20Formally, the regulator minimizes the virality of content conditional on the article containing misinformation, in mathematical notation, the regulator minimizes $\mathbb{E}_{max}^*|\mathbf{S}_v|\, \nu = M|$.","'What is the formal objective of the regulator in controlling misinformation virality, as expressed mathematically?'"
"21 In practice, certain demographic groups, such as users over 65 years old, appear more likely to accept misinformation without fact-checking, perhaps because of poor media interpretation skills (see Grinberg et al. (2019) and Guess et al. (2019)).","'Which demographic group is more likely to accept misinformation without fact-checking, and what potential reason does the context provide for this behavior?'"
"In Theorem 3, the threshold $r_p$ fully summarizes the extent to which misinformation will spread virally on social media. We next perform comparative statics for this threshold to understand the conditions under which the platform will create a filter bubble and propagate misinformation.",'What does the threshold $r_p$ in Theorem 3 reveal about the conditions under which social media platforms create filter bubbles and propagate misinformation?'
"Proposition 5. There exists $\rho^*<1$ such that: (a) If $\rho > \rho^*$ for any fixed sharing network, the provenance policy reduces the virality of misinformation. (b) If $\rho < \beta^*$ , the provenance policy has an ambiguous effect on the virality of misinformation.","'What is the critical threshold $\rho^*$, and how does it determine the effectiveness of the provenance policy in reducing misinformation virality?'"


ft_eval_dataset.num_rows: 25

,anchor,positive
0,6 Regulation\nOur analysis so far raises the n...,'What types of regulations does the paper sugg...
1,We consider the effects of these policies both...,'How does the paper model the regulator’s stra...
2,misinformation below a given threshold; (4) ne...,'How do network regulations aim to mitigate th...
3,and recommends articles to users with aligned ...,'How do algorithmic recommendations contribute...
4,"f misinformation, agents will be uncertain abo...",'How does misinformation affect agents' abilit...
5,information. We can also use this extended set...,'How does the probability of misinformation af...
6,ity of content. Because this article is unlike...,'How does increased homophily affect the sprea...
7,latform. Articles may be truthful and informat...,"'How do agents decide whether to share, ignore..."
8,"rk. Fully endogenizing reputational concerns, ...",'How might fully endogenizing reputational con...
9,"Nguyen, Nam P., Guanhua Yan, My T. Thai, and S...",'What strategies does the paper propose for co...


## Define training loss

In [0]:
if TRAINING_LOSS == 'MNRL':
    train_loss = sentence_transformers.losses.MultipleNegativesRankingLoss(model=embed_model)
elif TRAINING_LOSS == 'CSL':
    train_loss = sentence_transformers.losses.CosineSimilarityLoss(model=embed_model)
elif TRAINING_LOSS == 'CMNRL':
    train_loss = sentence_transformers.losses.CachedMultipleNegativesRankingLoss(
        model=embed_model,
        mini_batch_size=pow(2, 1)
    )
else:
    raise ValueError(f'Unsupported training loss: {TRAINING_LOSS}')
    
print(train_loss)


MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
    (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': 
True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 
'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
    (2): Normalize()
  )
  (cross_entropy_loss): CrossEntropyLoss()
)

## Perform fine-tuning

In [0]:
# remove distributed vars to avoid cluster issues
for var in ['RANK', 'WORLD_SIZE', 'LOCAL_RANK', 'MASTER_ADDR', 'MASTER_PORT']:
    os.environ.pop(var, None)

In [0]:
if RUN_TRAIN_MODEL:
    # https://huggingface.co/blog/train-sentence-transformers

    # tranining parameters
    tmp_dir = tempfile.TemporaryDirectory()
    base_model = sentence_transformers.SentenceTransformer(embed_model_name, device='cpu')

    args = sentence_transformers.training_args.SentenceTransformerTrainingArguments(
        auto_find_batch_size=False,
        data_seed=42,
        eval_steps=EVAL_STEPS_TRAINING,
        learning_rate=2e-5,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        output_dir=tmp_dir.name,
        seed=42,
        use_cache=False,
        use_cpu=True,
        warmup_steps=0,
    )

### Run training

In [0]:
if RUN_TRAIN_MODEL:

    class SentenceTransformerWrapper(mlflow.pyfunc.PythonModel):
        '''Wrapper for log the model in mlflow.'''

        def load_context(self, context):
            model_path = context.artifacts['model_path']
            self.model = sentence_transformers.SentenceTransformer(model_path)

        def predict(
            self,
            context: mlflow.pyfunc.PythonModelContext,
            model_input: List[str]
        ) -> List[List[float]]:
            embeddings = self.model.encode(model_input)
            return np.nan_to_num(embeddings, nan=0.0).tolist()

    try:
        trainer = sentence_transformers.SentenceTransformerTrainer(
            model=base_model,
            args=args,
            train_dataset=ft_train_dataset,
            eval_dataset=ft_eval_dataset,
            loss=train_loss,
        )

        print('Training the model ...')
        with mlflow.start_run() as run:

            trainer.train()

            local_model_path = 'fine_tuned_model'
            trainer.model.save(local_model_path)

            full_model_name = f'{CATALOG}.{SCHEMA}.{REGISTERED_FT_MODEL_NAME}'

            model_info = mlflow.pyfunc.log_model(
                name='model',
                python_model=SentenceTransformerWrapper(),
                artifacts={'model_path': local_model_path},
                registered_model_name=full_model_name,
                input_example=['Hello World!'],
            )

            model_uri = model_info.model_uri
        print('Done.')
        
        print(f'\nModel URI: {model_uri}')
        print(f'Unity Catalog model name: {full_model_name}')
        print(f'Local path: {local_model_path}')

    except Exception as e:
        print(f"Error:\n{e}")

## Inference with the fine-tuned model

In [0]:
def get_latest_model_version(model_name: str) -> int:
    mlflow_client = MlflowClient()
    model_version_infos = mlflow_client.search_model_versions(f"name = '{model_name}'")
    return max([int(i.version) for i in model_version_infos])

In [0]:
if TEST_FT_MODEL_LATEST_VERSION:
    latest_version = get_latest_model_version(model_name=f'{CATALOG}.{SCHEMA}.{REGISTERED_FT_MODEL_NAME}')
    print(f'Latest model version: {latest_version}')

In [0]:
if TEST_FT_MODEL_LATEST_VERSION:
    # deployed_version = 15
    latest_model_uc = f'models:/{CATALOG}.{SCHEMA}.{REGISTERED_FT_MODEL_NAME}/{latest_version}'
    try:
        pyfunc_model = mlflow.pyfunc.load_model(latest_model_uc)
    except Exception as e:
        print(f"Failed to load model at {latest_model_uc}: {e}")

    input_example = pyfunc_model.input_example

    loaded_model = mlflow.pyfunc.load_model(
        latest_model_uc
    )

    print(f'Input example:\n{input_example}\n')

    print('=== Base model ===')
    embeddings_base = embed_model.encode(input_example)
    print(f'Embeddings shape: {embeddings_base.shape}')
    print(list(embeddings_base[0][:10]))

    print('=== FT model ===')
    embeddings_ft = loaded_model.predict(input_example)
    print(f'Embeddings shape: {len(embeddings_ft)}')
    print(embeddings_ft[0][:10])

In [0]:
raise ValueError(f'Check whether the fine-tuned embedding model has already been deployed.')

## Load dataset for evaluation

In [0]:
prev_eval_set_df = spark.read.table(f'{CATALOG}.{SCHEMA}.{EVAL_SET_PREDS}')
display(prev_eval_set_df.show(5))

In [0]:
(
    prev_eval_set_df
    .groupBy('path')
    .count()
    .orderBy(desc('count'))
    .show(truncate=False)
)

## Deploy the fine-tuned model

- Estimated waiting time: ~50 min

- To deploy an endpoint for the fine-tuned model, use the UI or the commented code below.

In [0]:
# import secrets

# secret_key = secrets.token_hex(16)
# print(secret_key)

In [0]:
# # Setup secrets

# databricks secrets list-scopes
# databricks secrets create-scope genai_deploy
# databricks secrets put-secret genai_deploy GENAI_API_KEY

# ---

# databricks secrets put-secret --json '{ "scope": "finetuning_deploy", "key": "deploy_host", "string_value": "<host-name>"}
# databricks secrets put-secret --json '{ "scope": "finetuning_deploy", "key": "deploy_token", "string_value": "<token-value>"}

In [0]:
# from databricks.sdk.service.serving import EndpointCoreConfigInput 

# endpoint_config_dict = {
# 	'served_models': [{
# 		'model_name': REGISTERED_FT_MODEL_NAME,
# 		'model_version': latest_version,
# 		'scale_to_zero_enabled': True,
# 		'workload_size': 'Small',
# 		'enviroment_vars': {
# 			'DATABRICKS_TOKEN': '{{secrets/latest_model/deploy_host}}',
# 			'DATABRICKS_HOST': '{{secrets/latest_model/deploy_host}}'
# 		}
# 	}],
# 	'auto_capture_config': {
# 		'catalog_name': CATALOG,
# 		'schema_name': SCHEMA,
# 		'table_name_prefix': 'ft_embedding_model_gte_small'
# 	}
# }

# endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)
# print(endpoint_config)

In [0]:
# workspace_client = WorkspaceClient()
# print([endpoint for endpoint in workspace_client.list()])

# endpoint_name = REGISTERED_FT_MODEL_NAME
# db_host = DATABRICKS_HOST

# endpoint_url = f'{db_host}/ml/endpoints/{endpoint_name}'

# workspace_client.serving_endpoints.update_config_and_wait(name=endpoint_name, served_models=edpoint_config.served_models)

<img src='./imgs/serving_endpoint_ft_embedding.png'>

## Test the deployed endpoint for the fine-tuned model

In [0]:
%sql
SELECT ai_query(
  'ft_embedding_model_endpoint',
  'Hello World!'
)

"ai_query('ft_embedding_model_endpoint','Hello World!')"
